# UTeM - BAXI 3133: Neural Network (Semester 2, Sesi 2025/2026)
## Group Mini-Project: Cardiac Risk Stratification using Multilayer Perceptron (MLP)

This notebook contains the complete, production-ready, and fully commented Python code to build a Multilayer Perceptron (MLP) Artificial Neural Network (ANN) model to predict the presence or absence of heart disease (binary classification).

### CRISP-DM Methodology Stages Covered:
1. **Business Understanding**: Define the objective (stratify cardiac risk to detect heart disease).
2. **Data Understanding**: Review dataset features (12 features including continuous and nominal categorical columns).
3. **Data Preparation**: Handle missing values, perform one-hot encoding, train-test splitting, and standard scale features.
4. **Modeling**: Build, compile, and train a Keras Sequential MLP model with Dropout regularization.
5. **Evaluation**: Predict on the test set, output a Confusion Matrix and a Classification Report (Accuracy, Precision, Recall, F1-Score), and visualize performance.
6. **Deployment**: Store and prepare model outputs for clinical risk reporting.

### Step 1: Install & Import Libraries
Ensure we have all required deep learning and analytics libraries installed.

In [4]:
# Import core modules for analysis, modeling, and visualization
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

ModuleNotFoundError: No module named 'pandas'

### Step 2: DATA PREPROCESSING (CRISP-DM Phase 3)

In this phase, we:
1. Load the dataset (or generate the placeholder `heart_disease.csv` from raw records if running in the project directory).
2. Cast nominal categorical variables to string and perform one-hot encoding using `pd.get_dummies`.
3. Split variables into 80% training and 20% testing sets.
4. Apply `StandardScaler` to scale continuous numerical features, fitting the scaler only on the training set to prevent data leakage.

In [ ]:
# 1. Load or Generate the Dataset
csv_filename = 'heart_disease.csv'

# Fallback logic to check for Data/cardiac arrest dataset.csv if heart_disease.csv is not immediately present
if not os.path.exists(csv_filename):
    raw_path = os.path.join('..', 'Data', 'cardiac arrest dataset.csv')
    if not os.path.exists(raw_path):
        raw_path = os.path.join('Data', 'cardiac arrest dataset.csv')
        
    if os.path.exists(raw_path):
        print(f"Found raw dataset at {raw_path}. Mapping to UTeM 12-column spec...")
        df_raw = pd.read_csv(raw_path)
        
        # Rename according to spec
        df_raw = df_raw.rename(columns={
            'cp': 'chest_pain_type',
            'trestbps': 'resting_bp',
            'chol': 'cholesterol',
            'fbs': 'fasting_blood_sugar',
            'restecg': 'resting_ecg',
            'thalach': 'max_heart_rate',
            'exang': 'exercise_angina',
            'slope': 'st_slope'
        })
        
        # Filter to 12 target columns
        columns_to_keep = [
            'age', 'sex', 'chest_pain_type', 'resting_bp', 'cholesterol', 
            'fasting_blood_sugar', 'resting_ecg', 'max_heart_rate', 
            'exercise_angina', 'oldpeak', 'st_slope', 'target'
        ]
        df_raw = df_raw[columns_to_keep]
        
        # Scale chest_pain_type from 0-3 to 1-4 nominal values
        df_raw['chest_pain_type'] = df_raw['chest_pain_type'] + 1
        df_raw.to_csv(csv_filename, index=False)
        print(f"Placeholder '{csv_filename}' successfully generated.")
    else:
        # Create dummy placeholder data for testing if no file found
        print("No local dataset found. Creating synthetic heart_disease.csv placeholder...")
        np.random.seed(42)
        n_samples = 300
        synthetic_data = {
            'age': np.random.randint(29, 80, size=n_samples),
            'sex': np.random.randint(0, 2, size=n_samples),
            'chest_pain_type': np.random.randint(1, 5, size=n_samples),
            'resting_bp': np.random.randint(94, 200, size=n_samples),
            'cholesterol': np.random.randint(126, 564, size=n_samples),
            'fasting_blood_sugar': np.random.randint(0, 2, size=n_samples),
            'resting_ecg': np.random.randint(0, 3, size=n_samples),
            'max_heart_rate': np.random.randint(71, 202, size=n_samples),
            'exercise_angina': np.random.randint(0, 2, size=n_samples),
            'oldpeak': np.random.uniform(0.0, 6.2, size=n_samples),
            'st_slope': np.random.randint(0, 3, size=n_samples),
            'target': np.random.randint(0, 2, size=n_samples)
        }
        df_synthetic = pd.DataFrame(synthetic_data)
        df_synthetic.to_csv(csv_filename, index=False)
        print(f"Synthetic '{csv_filename}' successfully generated.")

# Read the clean data
df = pd.read_csv(csv_filename)
print(f"Loaded dataset shape: {df.shape}")
df.head()

In [ ]:
# 2. One-hot encoding for nominal variables ('chest_pain_type', 'resting_ecg', 'st_slope') using pandas
nominal_cols = ['chest_pain_type', 'resting_ecg', 'st_slope']
for col in nominal_cols:
    df[col] = df[col].astype(str)  # Cast to string to ensure nominal encoding

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

# Ensure boolean one-hot outputs are converted to numeric float32 values for Keras compatibility
encoded_columns = [col for col in df_encoded.columns if col not in ['target']]
for col in encoded_columns:
    if df_encoded[col].dtype == bool:
        df_encoded[col] = df_encoded[col].astype('float32')

print(f"Encoded dataset shape: {df_encoded.shape}")

# Separate features and target
X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

# 3. Split data into 80% training and 20% testing sets
# stratify=y preserves label distributions across splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Feature scaling of continuous numerical columns using StandardScaler
continuous_cols = ['age', 'resting_bp', 'cholesterol', 'max_heart_rate', 'oldpeak']
scaler = StandardScaler()

# Prevent data leakage: fit scaler ONLY on train set, transform both train and test sets
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(f"X_train shape: {X_train_scaled.shape}, X_test shape: {X_test_scaled.shape}")

### Step 3: MLP ARCHITECTURE & MODEL COMPILATION (CRISP-DM Phase 4)

We build a feedforward network with:
- An input layer matching the processed feature count.
- Two hidden Dense layers with ReLU activation functions.
- Dropout layers (0.2) following each hidden layer to prevent model overfitting.
- An output layer with 1 neuron and Sigmoid activation (outputs probability range 0.0 to 1.0).
- Optimization via the standard `Adam` optimizer, tracking binary loss (`binary_crossentropy`) and tracking prediction `accuracy`.

In [ ]:
# Define model dimension
input_dim = X_train_scaled.shape[1]

# Create Sequential architecture
model = Sequential([
    # Input layer matching processed feature count
    Input(shape=(input_dim,), name="input_layer"),
    
    # First Hidden layer: 64 neurons with ReLU
    Dense(64, activation='relu', name="hidden_layer_1"),
    Dropout(0.2, name="dropout_1"),
    
    # Second Hidden layer: 32 neurons with ReLU
    Dense(32, activation='relu', name="hidden_layer_2"),
    Dropout(0.2, name="dropout_2"),
    
    # Output layer: 1 neuron with Sigmoid activation for binary classification
    Dense(1, activation='sigmoid', name="output_layer")
])

# Model compilation
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Step 4: MODEL TRAINING & VISUALIZATION (CRISP-DM Phase 4)

We train the network for **100 epochs** with a **batch size of 32**. A validation split of 20% of the training dataset is monitored to observe loss and accuracy movements on unseen subsets during training.

In [ ]:
# Fit the model
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

### Step 5: Visualizing Training Curves

Using Matplotlib, we construct two separate line graphs:
1. **Training vs Validation Loss**
2. **Training vs Validation Accuracy**

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)

# Graph 1: Training vs Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(epochs_range, history.history['loss'], 'b-', label='Training Loss', linewidth=2)
plt.plot(epochs_range, history.history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
plt.title('Training vs Validation Loss across Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10)
plt.ylabel('Loss', fontsize=10)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Graph 2: Training vs Validation Accuracy
plt.figure(figsize=(10, 5))
plt.plot(epochs_range, history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
plt.plot(epochs_range, history.history['val_accuracy'], 'r--', label='Validation Accuracy', linewidth=2)
plt.title('Training vs Validation Accuracy across Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10)
plt.ylabel('Accuracy', fontsize=10)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Step 6: EVALUATION (CRISP-DM Phase 5)

We evaluate the model on the test dataset to assess final generalization performance:
1. Generate probabilistic predictions on the test set.
2. Classify values >= 0.5 as Heart Disease (Class 1) and values < 0.5 as Normal (Class 0).
3. Display classification scores (Accuracy, Precision, Recall, F1-Score).
4. Output a Seaborn heatmap representing the confusion matrix.

In [ ]:
# Generate predictions (probabilities)
y_pred_prob = model.predict(X_test_scaled)
# Convert to binary target classes using a 0.5 threshold
y_pred = (y_pred_prob >= 0.5).astype(int)

# Generate Confusion Matrix & Classification Report
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Normal (0)', 'Heart Disease (1)'])

print("Classification Report:")
print(report)

print("Confusion Matrix:")
print(cm)

# Plot Heatmap for professional report representation
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Normal (0)', 'Predicted Disease (1)'],
            yticklabels=['Actual Normal (0)', 'Actual Disease (1)'],
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Confusion Matrix Heatmap', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Predicted Label', fontsize=11, labelpad=10)
plt.ylabel('True Label', fontsize=11, labelpad=10)
plt.tight_layout()
plt.show()

---
# DOCUMENTATION SUPPORT FOR THE REPORT

## 1. Details on Deep Learning Functions Used
Copy this section into your **"Detail on DL functions"** report section:

- **`Sequential`**: A linear stack of layers in Keras. It allows you to build models layer-by-layer by grouping a stack of layers where each layer has exactly one input tensor and one output tensor.
- **`Input`**: A Keras layer used to instantiate a Keras tensor. It specifies the input shape (`input_dim`) matching the number of features of the preprocessed dataset, allowing the network to allocate weights properly.
- **`Dense`**: A regular deeply-connected neural network layer. Every neuron in a Dense layer receives input from all neurons in the previous layer. 
  - In the hidden layers, it applies the **`ReLU`** activation function ($f(x) = \max(0, x)$) to introduce non-linearity, enabling the model to learn complex relationships.
  - In the output layer, it has 1 neuron and applies the **`Sigmoid`** activation function ($f(x) = \frac{1}{1 + e^{-x}}$) to compress outputs into a probabilistic range $[0, 1]$ representing risk.
- **`Dropout`**: A regularization layer. It randomly sets input units to 0 with a frequency rate (0.2 or 20%) during training epochs, which forces the network to learn redundant representations, effectively mitigating overfitting.
- **`compile`**: Configures the model training process. It defines:
  - **`Adam`**: An adaptive stochastic gradient descent optimizer that computes individual adaptive learning rates for different parameters.
  - **`binary_crossentropy`**: The loss function used for binary classification. It measures the distance between the target distribution and predicted probabilities.
  - **`metrics=['accuracy']`**: The evaluation metric calculated during training and validation to observe the ratio of correct predictions over total predictions.
- **`fit`**: Trains the model for a fixed number of iterations (epochs) on a dataset. It splits data for validation, divides observations into batches, updates model weights via backpropagation, and logs training history metrics.
- **`predict`**: Feeds testing data through the trained network weights to compute output probability forecasts.

## 2. Pipeline Data Flow Logic for Flowchart Construction
Use this step-by-step logic layout to draw your project flowchart diagram:

1. **Start** (Beginning of the prediction workflow).
2. **Data Ingestion**: Load the `heart_disease.csv` raw tabular data file.
3. **Nominal Categorical Encoding**: 
   - Identify `chest_pain_type`, `resting_ecg`, and `st_slope` variables.
   - Convert these to category levels and apply pandas `get_dummies` to one-hot encode them.
4. **Train-Test Dataset Splitting**: Divide observations into an **80% Training Set** and a **20% Testing Set** (using stratification based on the target column).
5. **Feature Scaling (Standardization)**:
   - Calculate mean and standard deviation from training set continuous features (`age`, `resting_bp`, `cholesterol`, `max_heart_rate`, `oldpeak`).
   - Standardize training continuous variables using the computed statistics: $z = \frac{x - \mu}{\sigma}$.
   - Apply the exact same scaling parameters to the testing continuous features to prevent data leakage.
6. **Model Initialization**: Define Keras `Sequential` container.
7. **Layer Stack Setup**:
   - Feed inputs into **Hidden Layer 1** (64 nodes, ReLU activation) -> Apply **Dropout Regularization** (20%).
   - Feed outputs into **Hidden Layer 2** (32 nodes, ReLU activation) -> Apply **Dropout Regularization** (20%).
   - Feed outputs to **Output Layer** (1 node, Sigmoid activation).
8. **Compilation**: Bind architecture with **Adam Optimizer**, **Binary Crossentropy Loss**, and **Accuracy Metric**.
9. **Model Training**: Call `fit()` for **100 epochs**, using a **batch size of 32** and keeping aside a **20% validation split** to monitor overfitting.
10. **Loss & Accuracy Evaluation**: Plot learning curves across epochs. If satisfied, proceed. If overfitting occurs, tune dropout rates or units.
11. **Prediction & Binarization**: Call `predict()` on the scaled testing features. Binarize output probabilities using a $0.5$ decision threshold.
12. **Metrics Computation**: Generate a classification report (Accuracy, Precision, Recall, F1-Score) and Confusion Matrix.
13. **End** (Final stratified cardiac risk reports generated).